# Whisper large-v3 on Colab — locality clip transcription

**Why this notebook exists:** Whisper large-v3 needs a GPU. A Windows laptop without CUDA will silently downgrade to `medium` (the runner does this safely), but for the best benchmark we want the actual large-v3 numbers. Colab free tier (T4) gives us that.

**Workflow:**
1. Mount Google Drive
2. Point at the `recordings/` folder (uploaded there from your laptop)
3. Run Whisper on all 20 clips
4. Download the per-clip JSON outputs back to your laptop, drop into `results/whisper/`

**Runtime:** Set Runtime → Change runtime type → GPU (T4).

## 1. Setup

In [ ]:
!pip install -q openai-whisper jiwer python-Levenshtein metaphone rapidfuzz

In [ ]:
import torch
print('CUDA available:', torch.cuda.is_available())
print('Device:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU')

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

## 2. Configure paths

Upload the `recordings/` folder and `ground_truth.csv` to your Google Drive (e.g. under `MyDrive/asr-shootout/`).

In [ ]:
from pathlib import Path
PROJECT = Path('/content/drive/MyDrive/asr-shootout')
RECORDINGS = PROJECT / 'recordings'
GROUND_TRUTH = PROJECT / 'ground_truth.csv'
OUT_DIR = PROJECT / 'results' / 'whisper'
OUT_DIR.mkdir(parents=True, exist_ok=True)

audio_files = sorted(RECORDINGS.glob('*.m4a'))
print(f'Found {len(audio_files)} audio files')
for f in audio_files[:3]:
    print('  ', f.name)

## 3. Load Whisper large-v3

In [ ]:
import whisper, time
model = whisper.load_model('large-v3')
print('Loaded large-v3')

## 4. Transcribe all clips

In [ ]:
import json

results = []
for f in audio_files:
    t0 = time.perf_counter()
    out = model.transcribe(str(f), task='transcribe', fp16=True)
    latency_ms = (time.perf_counter() - t0) * 1000
    record = {
        'clip_id': f.stem,
        'filename': f.name,
        'model': 'whisper-large-v3',
        'transcript': out['text'].strip(),
        'language_detected': out.get('language'),
        'latency_total_ms': latency_ms,
        'raw_response': {
            'text': out['text'],
            'language': out.get('language'),
            'segments': [{k: s[k] for k in ('id','start','end','text') if k in s} for s in out.get('segments', [])],
        }
    }
    (OUT_DIR / f'{f.stem}.json').write_text(json.dumps(record, ensure_ascii=False, indent=2), encoding='utf-8')
    results.append(record)
    print(f'{f.stem:<55} {latency_ms:>6.0f} ms  ->  {record["transcript"][:80]}')

print(f'\nDone. {len(results)} transcripts saved to {OUT_DIR}')

## 5. Sanity check on locality recovery

Quick visual: for each clip, does the locality name appear in the transcript?

In [ ]:
import pandas as pd
gt = pd.read_csv(GROUND_TRUTH)
by_file = {r.filename: r for _, r in gt.iterrows()}

for r in results:
    expected = by_file[r['filename']].locality_canonical.lower()
    got = r['transcript'].lower()
    hit = '✓' if expected in got else '✗'
    print(f'{hit}  {r["clip_id"]:<55}  expected: {expected}')

## 6. Download results back to laptop

After this runs, go to `MyDrive/asr-shootout/results/whisper/` in Drive, download the folder, and place its contents into `results/whisper/` in your local project. The orchestrator and analysis scripts will pick them up automatically.